In [1]:
"""
Build train / test datasets from train.csv with a stratified 80/20 split
(DATA5925 / EcoStat Modelling).

Split logic (exactly as requested):
  For every species, the positive rows (pres.abs == 1) and the negative rows
  (pres.abs == 0) are split SEPARATELY, 80% to train and 20% to test.

  Example: species "A" has 5 positives and 5 negatives ->
           4 positive train + 1 positive test, 4 negative train + 1 negative test.

This preserves each species' presence rate in both partitions and ensures every
species (even very rare ones) appears in both train and test where possible.

Edge cases:
  * A (species, class) group with a single row goes entirely to train (it cannot
    appear in both); a warning is printed.
  * Rounding keeps at least one row on each side whenever the group has >= 2 rows.
    e.g. 3 positives -> 2 train + 1 test.

Note on leakage: this is a ROW-level split, so the eight rows of one plot may be
spread across train and test. That is the natural reading of the request here. If
you later want the leakage-free spatial split for model evaluation, split by `plot`
instead (see xgboost_species.py). The two serve different purposes.

Outputs:
  train_split.csv, test_split.csv  (all original columns preserved)

Usage:
    python split_train_test.py --data train.csv --test-size 0.2
"""

'\nBuild train / test datasets from train.csv with a stratified 80/20 split\n(DATA5925 / EcoStat Modelling).\n\nSplit logic (exactly as requested):\n  For every species, the positive rows (pres.abs == 1) and the negative rows\n  (pres.abs == 0) are split SEPARATELY, 80% to train and 20% to test.\n\n  Example: species "A" has 5 positives and 5 negatives ->\n           4 positive train + 1 positive test, 4 negative train + 1 negative test.\n\nThis preserves each species\' presence rate in both partitions and ensures every\nspecies (even very rare ones) appears in both train and test where possible.\n\nEdge cases:\n  * A (species, class) group with a single row goes entirely to train (it cannot\n    appear in both); a warning is printed.\n  * Rounding keeps at least one row on each side whenever the group has >= 2 rows.\n    e.g. 3 positives -> 2 train + 1 test.\n\nNote on leakage: this is a ROW-level split, so the eight rows of one plot may be\nspread across train and test. That is the n

In [2]:
import argparse
import numpy as np
import pandas as pd

TARGET = "pres.abs"
SPECIES_COL = "Species"
RANDOM_STATE = 42


def split_counts(n, test_size):
    """Number of (train, test) rows for a group of size n."""
    if n <= 1:
        return n, 0  # single row -> train only
    n_test = int(round(n * test_size))
    n_test = max(1, min(n_test, n - 1))  # keep at least one row on each side
    return n - n_test, n_test


def stratified_split(df, test_size, random_state):
    rng = np.random.default_rng(random_state)
    train_parts, test_parts, summary = [], [], []

    # Group by species AND class, then split each group independently.
    for (species, cls), group in df.groupby([SPECIES_COL, TARGET], sort=True):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)  # shuffle within the group for a random split
        n_train, n_test = split_counts(len(idx), test_size)

        train_idx, test_idx = idx[:n_train], idx[n_train:]
        train_parts.append(df.loc[train_idx])
        test_parts.append(df.loc[test_idx])

        summary.append({
            "Species": species,
            "class": "positive" if cls == 1 else "negative",
            "total": len(idx),
            "train": n_train,
            "test": n_test,
        })

        if len(idx) == 1:
            print(f"  [warning] {species} / "
                  f"{'positive' if cls == 1 else 'negative'} has only 1 row "
                  f"-> placed in TRAIN only.")

    train_df = pd.concat(train_parts).sample(frac=1, random_state=random_state)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=random_state)
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True), pd.DataFrame(summary)


def main():
    parser = argparse.ArgumentParser(description="Stratified per-species per-class split.")
    parser.add_argument("--data", default="train.csv")
    parser.add_argument("--test-size", type=float, default=0.2)
    parser.add_argument("--train-out", default="train_split.csv")
    parser.add_argument("--test-out", default="test_split.csv")
    args, _ = parser.parse_known_args()  # notebook-safe

    print(f"Loading {args.data} ...")
    df = pd.read_csv(args.data)
    for col in (TARGET, SPECIES_COL):
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Columns: {list(df.columns)}")

    print(f"  rows={len(df)}, species={df[SPECIES_COL].nunique()}, "
          f"overall presence rate={df[TARGET].mean():.4f}\n")

    train_df, test_df, summary = stratified_split(df, args.test_size, RANDOM_STATE)

    # Per-species / per-class breakdown so the split is easy to verify.
    print("Split summary (rows per species and class):")
    print(summary.to_string(index=False))

    print(f"\nTotal -> train: {len(train_df)} rows "
          f"(presence rate {train_df[TARGET].mean():.4f}), "
          f"test: {len(test_df)} rows "
          f"(presence rate {test_df[TARGET].mean():.4f})")

    train_df.to_csv(args.train_out, index=False)
    test_df.to_csv(args.test_out, index=False)
    print(f"\nSaved {args.train_out} and {args.test_out}")


if __name__ == "__main__":
    main()

Loading train.csv ...
  rows=5128, species=8, overall presence rate=0.0634

Split summary (rows per species and class):
                   Species    class  total  train  test
          Cacophis kreftii negative    621    497   124
          Cacophis kreftii positive     20     16     4
   Calyptotis scutirostrum negative    591    473   118
   Calyptotis scutirostrum positive     50     40    10
Coeranoscincus reticulatus negative    537    430   107
Coeranoscincus reticulatus positive    104     83    21
           Egernia mcpheei negative    629    503   126
           Egernia mcpheei positive     12     10     2
         Eulamprus murrayi negative    638    510   128
         Eulamprus murrayi positive      3      2     1
    Ophioscincus truncatus negative    527    422   105
    Ophioscincus truncatus positive    114     91    23
   Pseudechis porphyricaus negative    627    502   125
   Pseudechis porphyricaus positive     14     11     3
         Saltuarius swaini negative    6